# Visualización de las últimas 20 posiciones del Ground Truth
Muestra tablero + BSPs activos para las últimas 20 posiciones del dataset de 200 juegos.

In [1]:
import numpy as np
import sys
from pathlib import Path

project_root = Path('../../..').resolve()
sys.path.insert(0, str(project_root))

# Cargar datos
boards_data = np.load(project_root / 'sae' / 'metrics' / '02_data' / 'board_states_1games.npz')
boards_all  = boards_data['boards']   
colors_all  = boards_data['colors']  

bsp_gt    = np.load(project_root / 'sae' / 'metrics' / '02_data' / 'bsp_ground_truth_1games.npy')       # (11800, 198)
bsp_names = np.load(project_root / 'sae' / 'metrics' / '02_data' / 'bsp_ground_truth_1games.names.npy', allow_pickle=True)

print(f'boards_all : {boards_all.shape}')
print(f'bsp_gt     : {bsp_gt.shape}')
print(f'n_bsps     : {len(bsp_names)}')

boards_all : (1, 59, 8, 8)
bsp_gt     : (59, 326)
n_bsps     : 326


In [2]:
# Últimas 20 posiciones (último juego, últimos 20 movimientos)
last20_bsp    = bsp_gt[-20:]       # (20, 198)
last20_boards = boards_all.reshape(-1, 8, 8)[-20:]   
last20_colors = colors_all.reshape(-1)[-20:]        

print(f'last20_bsp    : {last20_bsp.shape}')
print(f'last20_boards : {last20_boards.shape}')
print(f'last20_colors : {last20_colors}')

last20_bsp    : (20, 326)
last20_boards : (20, 8, 8)
last20_colors : [-1  1 -1  1 -1  1 -1  1 -1  1 -1  1 -1  1 -1  1 -1  1 -1  1]


In [7]:
def print_board(board, color, pos_idx):
    color_str = 'Negro (1)' if color == 1 else 'Blanco (-1)'
    print(f'\n=== Posición {pos_idx+1} | Turno: {color_str} ===')
    print('  1 2 3 4 5 6 7 8')
    for i, row in enumerate(board):
        symbols = []
        for val in row:
            if val == 1:   symbols.append('●')
            elif val == -1: symbols.append('○')
            else:           symbols.append('·')
        print(f'{"abcdefgh"[i]} {" ".join(symbols)}')
    black = int((board == 1).sum())
    white = int((board == -1).sum())
    print(f'  Negras: {black}  Blancas: {white}  Total: {black+white}')


def print_active_bsps(bsp_row, bsp_names):
    active = [bsp_names[i] for i, v in enumerate(bsp_row) if v == 1]
    print(f'  BSPs activos ({len(active)}): {" ".join(active) if active else "ninguno"}')


# Mostrar las 20 posiciones
for i in range(20):
    print_board(last20_boards[i], last20_colors[i], i)
    print_active_bsps(last20_bsp[i], bsp_names)


=== Posición 1 | Turno: Blanco (-1) ===
  1 2 3 4 5 6 7 8
a · ● ○ · ● ● ○ ·
b ○ ● ○ ● ● ○ ○ ○
c · ● ● ● ○ ● ○ ●
d ○ ● ○ ○ ● ● ● ·
e ○ ○ ○ ● ○ ○ · ·
f ○ ○ ○ ○ · ○ · ·
g ○ ○ ○ ○ · ○ · ·
h · ○ · · · · · ·
  Negras: 16  Blancas: 28  Total: 44
  BSPs activos (109): BSPA10 BSPA22 BSPA31 BSPA40 BSPA52 BSPA62 BSPA71 BSPA80 BSPB11 BSPB22 BSPB31 BSPB42 BSPB52 BSPB61 BSPB71 BSPB81 BSPC10 BSPC22 BSPC32 BSPC42 BSPC51 BSPC62 BSPC71 BSPC82 BSPD11 BSPD22 BSPD31 BSPD41 BSPD52 BSPD62 BSPD72 BSPD80 BSPE11 BSPE21 BSPE31 BSPE42 BSPE51 BSPE61 BSPE70 BSPE80 BSPF11 BSPF21 BSPF31 BSPF41 BSPF50 BSPF61 BSPF70 BSPF80 BSPG11 BSPG21 BSPG31 BSPG41 BSPG50 BSPG61 BSPG70 BSPG80 BSPH10 BSPH21 BSPH30 BSPH40 BSPH50 BSPH60 BSPH70 BSPH80 BSPA2B BSPA3W BSPA5B BSPA6B BSPA7W BSPB1W BSPB2B BSPB3W BSPB4B BSPB5B BSPB6W BSPB7W BSPB8W BSPC2B BSPC3B BSPC4B BSPC5W BSPC6B BSPC7W BSPC8B BSPD1W BSPD2B BSPD3W BSPD4W BSPD5B BSPD6B BSPD7B BSPE1W BSPE2W BSPE3W BSPE4B BSPE5W BSPE6W BSPF1W BSPF2W BSPF3W BSPF4W BSPF6W BSPG1W BSPG2W BSPG3W BSP

In [8]:
# Resumen: tabla BSP x posición (20x198) — solo BSPs de piezas (no vacías)
piece_mask = np.array([len(n) == 6 and n.startswith('BSP') and not n.endswith('0') for n in bsp_names])
piece_names = bsp_names[piece_mask]          # 128 BSPs
last20_piece_bsp = last20_bsp[:, piece_mask] # (20, 128)

print('Matriz BSPs de piezas (20 posiciones x 128 BSPs)')
print(f'Shape: {last20_piece_bsp.shape}')
print(f'Activos por posición: {last20_piece_bsp.sum(axis=1).tolist()}')
print(f'Posiciones activas por BSP (top 10):')
counts = last20_piece_bsp.sum(axis=0)
top10  = np.argsort(counts)[::-1][:10]
for idx in top10:
    print(f'  {piece_names[idx]}: {int(counts[idx])}/20 posiciones')

Matriz BSPs de piezas (20 posiciones x 128 BSPs)
Shape: (20, 256)
Activos por posición: [88, 90, 92, 94, 96, 98, 100, 102, 104, 106, 108, 110, 112, 114, 116, 118, 120, 122, 124, 126]
Posiciones activas por BSP (top 10):
  BSPG1W: 20/20 posiciones
  BSPF4W: 20/20 posiciones
  BSPE3W: 20/20 posiciones
  BSPE1W: 20/20 posiciones
  BSPE2W: 20/20 posiciones
  BSPF2W: 20/20 posiciones
  BSPF1W: 20/20 posiciones
  BSPA5B: 20/20 posiciones
  BSPB1W: 20/20 posiciones
  BSPB3W: 20/20 posiciones


In [9]:
import pandas as pd

piece_mask = np.array([len(n) == 6 and n.startswith('BSP') and not n.endswith('0') for n in bsp_names])
piece_names = bsp_names[piece_mask]
last20_piece_bsp = last20_bsp[:, piece_mask]  # (20, 128)

df = pd.DataFrame(
    last20_piece_bsp,
    columns=piece_names,
    index=[f'Pos_{i}' for i in range(20)]
)

print(f'DataFrame de las 20 posiciones:')
print('=' * 60)
print(df)

DataFrame de las 20 posiciones:
        BSPA11  BSPA12  BSPA21  BSPA22  BSPA31  BSPA32  BSPA41  BSPA42  \
Pos_0        0       0       0       1       1       0       0       0   
Pos_1        0       0       1       0       0       1       0       0   
Pos_2        0       0       0       1       1       0       0       0   
Pos_3        0       0       1       0       0       1       0       0   
Pos_4        0       0       0       1       1       0       1       0   
Pos_5        0       0       1       0       0       1       0       1   
Pos_6        0       0       0       1       1       0       1       0   
Pos_7        0       0       1       0       0       1       0       1   
Pos_8        0       0       0       1       1       0       1       0   
Pos_9        0       0       1       0       0       1       0       1   
Pos_10       0       0       0       1       1       0       1       0   
Pos_11       0       0       1       0       0       1       0       1   
Pos_12

In [12]:
BSP_OBJETIVO = "BSPH8B"

print(f'Ground truth de {BSP_OBJETIVO} en los 20 tableros:')
print('=' * 40)
print(df[[BSP_OBJETIVO]])

Ground truth de BSPH8B en los 20 tableros:
        BSPH8B
Pos_0        0
Pos_1        0
Pos_2        0
Pos_3        0
Pos_4        0
Pos_5        0
Pos_6        0
Pos_7        0
Pos_8        0
Pos_9        0
Pos_10       0
Pos_11       1
Pos_12       1
Pos_13       1
Pos_14       1
Pos_15       1
Pos_16       1
Pos_17       1
Pos_18       1
Pos_19       1
